In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
!pip install -U -q edgartools

In [ ]:
!pip install -U -q yfinance

In [ ]:
import edgar
import pandas as pd
import yfinance as yf

from public_power_backend.constants import OUTPUT_DIR

In [ ]:
edgar.set_identity("katherine.lamb@catalyst.coop")

In [ ]:
ious_df = pd.read_parquet(OUTPUT_DIR / "data_warehouse/pudl_utilities.parquet")

In [ ]:
ious_df["ticker"] = ious_df["central_index_key"].apply(
    lambda x: edgar.Company(x).get_ticker() if pd.notnull(x) else None
)

In [ ]:
ious_df["owner_company_ticker"] = ious_df["owner_company_central_index_key"].apply(
    lambda x: edgar.Company(x).get_ticker() if pd.notnull(x) else None
)

Use yfinance to get market cap and share price from most recent quarter.

In [ ]:
def get_shares_outstanding(ticker):
    return yf.Ticker(ticker).info["sharesOutstanding"]


def get_share_price(ticker, last_quarter_close_date="2025-06-30"):
    return yf.Ticker(ticker).history(start=last_quarter_close_date)["Close"].iloc[0]


def get_trailing_eps_and_pe(ticker):
    if ticker is None:
        return None
    trailing_eps = yf.Ticker(ticker).info["trailingEps"]
    trailing_pe = get_share_price(ticker) / trailing_eps
    return trailing_eps, trailing_pe


def get_market_cap_and_share_price(ticker):
    """Get market cap and share price at the end of the most recent quarter."""
    if ticker is None:
        return None
    shares = get_shares_outstanding(ticker)
    share_price = get_share_price(ticker)
    market_cap = shares * share_price
    return market_cap, share_price

In [ ]:
ious_df[["market_cap_quarter_end", "share_price_quarter_end"]] = (
    ious_df["owner_company_ticker"]
    .apply(get_market_cap_and_share_price)
    .apply(pd.Series)
)

In [ ]:
ious_df["market_cap_quarter_end"].isnull().value_counts()

In [ ]:
ious_df["owner_company_ticker"].isnull().value_counts()

In [ ]:
ious_df[["trailing_earnings_per_share", "trailing_pe_ratio_quarter_end"]] = (
    ious_df["owner_company_ticker"].apply(get_trailing_eps_and_pe).apply(pd.Series)
)

In [ ]:
ious_df["trailing_earnings_per_share"].isnull().value_counts()

In [ ]:
ious_df["trailing_pe_ratio_quarter_end"].isnull().value_counts()

Get institutional holder data

In [ ]:
def get_top_institutional_holders(ticker):
    stock = yf.Ticker(ticker)
    institutional_holders = stock.institutional_holders
    # major_holders = stock.major_holders
    if institutional_holders is None:
        return None
    top_holders = institutional_holders.sort_values(by="pctHeld", ascending=False).head(
        3
    )
    top_holders["ticker"] = ticker
    return top_holders

In [ ]:
holder_df_list = (
    ious_df["owner_company_ticker"]
    .dropna()
    .apply(get_top_institutional_holders)
    .tolist()
)

In [ ]:
len(holder_df_list)

In [ ]:
holder_df = pd.concat(holder_df_list, ignore_index=True)

In [ ]:
# we have duplicates because some companies have the same owner companies
holder_df = holder_df.drop_duplicates()

In [ ]:
holder_df = holder_df.rename(
    columns={
        "Holder": "institutional_holder_name",
        "Shares": "shares_millions",
        "pctHeld": "pct_held",
    }
)

In [ ]:
holder_df["holder_rank"] = holder_df.groupby("ticker").cumcount() + 1

In [ ]:
holder_df.holder_rank.max()

In [ ]:
holder_df

In [ ]:
holder_wide = holder_df.pivot(
    index="ticker",
    columns="holder_rank",
    values=["institutional_holder_name", "shares_millions", "pct_held"],
)

In [ ]:
holder_wide

In [ ]:
holder_wide.columns = [f"{col}_{rank}" for col, rank in holder_wide.columns]

In [ ]:
holder_wide = holder_wide[
    [
        "institutional_holder_name_1",
        "shares_millions_1",
        "pct_held_1",
        "institutional_holder_name_2",
        "shares_millions_2",
        "pct_held_2",
        "institutional_holder_name_3",
        "shares_millions_3",
        "pct_held_3",
    ]
]

In [ ]:
ious_df = ious_df.merge(
    holder_wide, how="left", left_on="owner_company_ticker", right_index=True
)

In [ ]:
ious_df.to_parquet(OUTPUT_DIR / "data_mart/owner_conditions.parquet")

In [ ]:
ious_df.to_csv("owner_conditions.csv")

SCRATCH: Use polygon to get owner company market cap from 2024 averaged over the year.

In [ ]:
API_KEY = os.environ["POLYGON_API_KEY"]

In [ ]:
import time

In [ ]:
def get_shares_outstanding(ticker):
    url = f"https://api.polygon.io/v3/reference/tickers/{ticker}"
    resp = requests.get(url, params={"apiKey": API_KEY}).json()
    return resp.get("results", {}).get("weighted_shares_outstanding")


def get_daily_prices(ticker, start="2024-01-01", end="2024-12-31"):
    url = f"https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}"
    resp = requests.get(url, params={"apiKey": API_KEY}).json()
    return resp.get("results", [])


def get_eps(ticker, year="2024"):
    url = "https://api.polygon.io/vX/reference/financials"
    resp = requests.get(
        url, params={"ticker": ticker, "apiKey": API_KEY, "timeframe": "annual"}
    ).json()
    time.sleep(12)
    results = resp.get("results", [])
    # filter for 2024 annual EPS if available
    eps_values = [
        r["financials"]["income_statement"]
        .get("diluted_earnings_per_share", {})
        .get("value")
        for r in results
        if r.get("fiscal_year") == year
    ]
    if not eps_values:
        return None
    return eps_values[0]  # take first available annual EPS


def get_market_cap_and_share_price(ticker):
    """Get avg market cap and share price."""
    if ticker is None:
        return None
    shares = get_shares_outstanding(ticker)
    time.sleep(12)
    if not shares:
        return None
    daily = get_daily_prices(ticker)
    time.sleep(12)
    closes = [bar["c"] for bar in daily if "c" in bar]
    avg_share_price = sum(closes) / len(closes) if closes else None
    if not daily:
        return None
    caps = [bar["c"] * shares for bar in daily if "c" in bar]
    avg_market_cap = sum(caps) / len(caps)
    return avg_market_cap, avg_share_price

In [ ]:
ious_df[["avg_market_cap", "avg_share_price"]] = (
    ious_df["owner_company_ticker"]
    .apply(get_market_cap_and_share_price)
    .apply(pd.Series)
)

In [ ]:
ious_df.avg_market_cap.isnull().value_counts()

In [ ]:
ious_df.owner_company_ticker.isnull().value_counts()

In [ ]:
ious_df["diluted_earnings_per_share"] = (
    ious_df["owner_company_ticker"].apply(get_eps).apply(pd.Series)
)

In [ ]:
ious_df["diluted_earnings_per_share"].isnull().value_counts()

In [ ]:
ious_df["pe_ratio"] = ious_df["avg_share_price"] / ious_df["diluted_earnings_per_share"]

In [ ]:
ious_df.to_csv("owner_conditions.csv")

In [ ]:
ious_df

In [ ]:
ious_df

In [ ]:
ious_df = ious_df.merge(
    holder_wide[
        [
            "institutional_holder_name_1",
            "shares_millions_1",
            "pct_held_1",
            "institutional_holder_name_2",
            "shares_millions_2",
            "pct_held_2",
            "institutional_holder_name_3",
            "shares_millions_3",
            "pct_held_3",
        ]
    ],
    how="left",
    right_index=True,
    left_on="owner_company_ticker",
)

In [ ]:
ious_df.to_csv("holder_test.csv")

SCRATCH: Get institutional holder data with SEC-API

In [ ]:
search_params = {
    "query": "formType:13F-HR AND periodOfReport:2024-12-31 AND holdings.ticker:AAPL",
    "sort": [{"periodOfReport": {"order": "desc"}}],
    # "size": 100 # Adjust to get more results, up to 200
}

response = requests.post(
    f"https://api.sec-api.io?token={SECIO_API_KEY}", json=search_params
)

if response.status_code == 200:
    filings = response.json().get("filings", [])

    # Process the filings to find the largest holders
    holders_data = []
    for filing in filings:
        for holding in filing.get("holdings", []):
            holders_data.append(
                {
                    "managerName": filing.get("managerName"),
                    "value": holding.get("value"),
                    "shares": holding.get("shares"),
                }
            )

    # Sort and display the top holders
    top_holders = sorted(holders_data, key=lambda x: x["value"], reverse=True)

    for holder in top_holders[:10]:  # Get the top 10
        print(f"Manager: {holder['managerName']}, Holding Value: ${holder['value']:,}")

else:
    print(f"Error: {response.status_code}, {response.text}")

In [ ]:
def get_top_institutional_holders(ticker, start=0, api_key=SECIO_API_KEY):
    """Fetch top institutional holders for a company CIK from SEC-API.io 13F filings.
    Returns list of dicts with name, shares, and pct.
    """
    url = f"https://api.sec-api.io?token={api_key}"
    query = {
        "query": {
            "query_string": {
                # "query": f"holdings.ticker:{ticker} AND formType:13F-HR AND filedAt:[2024-01-01 TO 2025-03-01]"
                "query": 'formType:"13F-HR" AND NOT formType:"13F-HR/A" AND periodOfReport:"2024-12-31"',
            }
        },
        "from": start,
        "size": 200,
        "sort": [{"periodOfReport": {"order": "desc"}}],
    }

    r = requests.post(url, json=query)
    if r.status_code != 200:
        return []
    res = r.json()
    if "filings" not in res or len(res["filings"]) == 0:
        return []

    holders_data = []
    for filing in res["filings"]:
        filed_at = filing["filedAt"]
        holder_name = filing.get("companyName")
        for holding in filing.get("holdings", []):
            if holding.get("ticker") == ticker:
                # pct = (shares / shares_outstanding * 100) if shares_outstanding else None
                holders_data.append(
                    {
                        "holder_name": holder_name,
                        "filing_date": filed_at,
                        "value": holding.get("value"),
                        "shares": holding["shrsOrPrnAmt"]["sshPrnamt"],
                    }
                )
                break
    return holders_data

In [ ]:
holders_data = get_top_institutional_holders("AEE")

In [ ]:
ious_df

In [ ]:
len(holders_data)

In [ ]:
def get_all_holders(ticker):
    start = 0
    holders_df = pd.DataFrame()
    while start < 10000:
        holders_data = get_top_institutional_holders(ticker, start)
        if len(holders_data) == 0:
            break
        df = pd.DataFrame(holders_data)
        df["filing_date"] = pd.to_datetime(df["filing_date"])
        df = (
            df.sort_values(by="filing_date", ascending=False)
            .groupby("holder_name", as_index=False)
            .first()
        )
        df = df.sort_values(by="shares", ascending=False).head(3)
        holders_df = pd.concat([holders_df, df])
        start = start + 200
    return holders_df

In [ ]:
pnw_df = get_all_holders("PNW")

In [ ]:
pnw_df.sort_values(by="shares", ascending=False)

In [ ]:
holders_df = pd.DataFrame(holders_data)

In [ ]:
holders_df["holder_name"] = holders_df["holder_name"].str.lower()

In [ ]:
holders_df["filing_date"] = pd.to_datetime(holders_df["filing_date"])
holders_df = (
    holders_df.sort_values(by="filing_date", ascending=False)
    .groupby("holder_name", as_index=False)
    .first()
)

In [ ]:
holders_df.sort_values(by="value", ascending=False).head(3)

In [ ]:
holders_df[holders_df.holder_name.str.contains("black")]

In [ ]:
holders

In [ ]:
ious_df.head(3)

In [ ]:
def get_top_institutional_holders(ticker, api_key=FMP_API_KEY, top_n=3):
    """Fetch top institutional holders for a ticker from FMP.
    Returns a list of dicts with name, shares, and pct.
    """
    url = f"https://financialmodelingprep.com/api/v4/institutional-ownership/symbol-ownership?symbol={ticker}&apikey={api_key}"

    r = requests.get(url)
    return r
    if r.status_code != 200:
        return []
    data = r.json()
    if not isinstance(data, list) or len(data) == 0:
        return []
    # Sort by shares and take top N
    top_holders = sorted(data, key=lambda x: x.get("shares", 0), reverse=True)[:top_n]
    return top_holders

In [ ]:
get_top_institutional_holders("MRK")

In [ ]:
url = f"https://financialmodelingprep.com/api/v4/institutional-holder/0001364742?apikey={FMP_API_KEY}"
r = requests.get(url)

In [ ]:
r

In [ ]:
ious_df.head(2)["owner_company_ticker"].apply(get_top_institutional_holders)

In [ ]:
[
    {
        "name": h.get("investorName"),
        "shares": h.get("shares"),
        "pct": h.get("percentage"),
    }
    for h in top_holders
]